# Toy MLIP Reliability Demo: 从势能面拟合到 OOD 诊断

**AI4S 公开课实战：训练一个 toy MLIP，并判断它什么时候不可信**

机器学习势函数直接学习从原子种类和坐标到能量的映射。模型输出能量后，可以通过能量对坐标的梯度得到原子力。

在真实材料模拟中，MLIP 的难点往往不只是把 ID 测试误差做低，而是判断它在新温压、新缺陷、新反应路径等训练数据覆盖之外的区域是否仍然可信。本实战用一个三维水分子 toy system，把这个可靠性诊断流程压缩到 15–20 分钟内完成。

## 学习目标

1. 训练一个轻量神经网络势函数，学习构型到能量和力的映射。
2. 检查能量旋转不变性与力旋转等变性，理解物理对称性为什么是 sanity check。
3. 比较 ID / OOD 误差，并用 committee disagreement 与数据回流模拟主动学习闭环。

本 notebook 不使用真实 DFT 数据，而是用解析 toy potential 生成“伪 DFT”能量和力标签。目标不是追求最高精度，而是理解 MLIP 可靠性诊断的基本工作流。



## Section 1. 数据模块：生成三维水分子 toy 势能面

这一节先构造一个可控的“伪 DFT”世界。每个构型是一个三维水分子，能量由两个 O-H 键长、H-O-H 角度和 H-H 排斥项决定，参考力由能量自动微分得到。

我们会生成三类数据视角：训练用的 ID 构型、接近平衡的 ID 测试构型，以及四类 OOD 构型。后面的所有可靠性分析，都建立在“训练数据覆盖了哪里、没有覆盖哪里”这个问题上。

这里的 OOD 指 out-of-distribution，也就是训练分布之外的构型。本 demo 里四类 OOD 分别对应：

- **OOD_highT**：键长和键角都有更大扰动，模拟高温下更剧烈的热振动。
- **OOD_stretch**：一个 O-H 键被明显拉长，模拟拉伸、断键前后的构型。
- **OOD_compress**：一个 O-H 键被明显压缩，模拟原子距离过近的高排斥区域。
- **OOD_angle**：H-O-H 角度明显偏离平衡值，模拟模型没有充分见过的弯曲构型。



先导入依赖、设置随机种子和绘图参数。这样每次运行都会从同一组随机数开始，便于复现实验结果。




In [ ]:
import os
import math
import random
import json
from pathlib import Path
import base64
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as widgets
import py3Dmol
from IPython.display import display, clear_output
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
DTYPE = torch.float32
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "Arial",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "figure.titleweight": "bold",
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

# Keep model colors consistent across all plots. These two colors are taken from the
# configuration-space scatter palette: blue for RawCoordNet, green for InvariantFeatureNet.
MODEL_COLORS = {
    "RawCoordNet": "#1f77b4",
    "InvariantFeatureNet": "#2ca02c",
}

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    """Save a classroom preview figure under outputs/figures."""
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path










这里定义“伪 DFT”势能面：水分子的能量由两个 O-H 键长、H-O-H 角度和 H-H 排斥项决定，参考力由能量对坐标的自动微分得到。




In [ ]:
# Toy potential parameters
R0 = 1.0
THETA0 = math.radians(104.5)
D_E = 5.0
A_MORSE = 2.5
K_ANGLE = 1.5

def compute_internal_coords_torch(coords, sort_h=False):
    """Compute r1, r2, theta from coords with shape (..., 3, 3)."""
    o = coords[..., 0, :]
    h1 = coords[..., 1, :]
    h2 = coords[..., 2, :]
    v1 = h1 - o
    v2 = h2 - o
    r1 = torch.linalg.norm(v1, dim=-1).clamp_min(1e-8)
    r2 = torch.linalg.norm(v2, dim=-1).clamp_min(1e-8)
    dot = (v1 * v2).sum(dim=-1)
    cross = torch.linalg.cross(v1, v2, dim=-1)
    cross_norm = torch.linalg.norm(cross, dim=-1)
    theta = torch.atan2(cross_norm, dot)
    if sort_h:
        rr = torch.sort(torch.stack([r1, r2], dim=-1), dim=-1).values
        return rr[..., 0], rr[..., 1], theta
    return r1, r2, theta


def toy_energy_torch(coords):
    """Reference toy potential energy for 3D water-like structures."""
    r1, r2, theta = compute_internal_coords_torch(coords, sort_h=False)
    morse1 = D_E * (1.0 - torch.exp(-A_MORSE * (r1 - R0)))**2
    morse2 = D_E * (1.0 - torch.exp(-A_MORSE * (r2 - R0)))**2
    angle = K_ANGLE * (theta - THETA0)**2
    rhh = torch.linalg.norm(coords[..., 1, :] - coords[..., 2, :], dim=-1).clamp_min(1e-8)
    repulsion = 0.02 / (rhh**6 + 1e-6)
    return morse1 + morse2 + angle + repulsion


def reference_energy_and_forces(coords_np):
    """Compute pseudo-DFT energy and forces with torch autograd."""
    coords = torch.tensor(coords_np, dtype=DTYPE, requires_grad=True)
    energy = toy_energy_torch(coords)
    grad = torch.autograd.grad(energy.sum(), coords)[0]
    forces = -grad
    return energy.detach().numpy(), forces.detach().numpy()


def _sample_angle_ood(n):
    """Sample low-angle and high-angle OOD structures."""
    half = n // 2
    low = np.random.uniform(np.deg2rad(55), np.deg2rad(80), size=half)
    high = np.random.uniform(np.deg2rad(140), np.deg2rad(170), size=n - half)
    theta = np.concatenate([low, high])
    np.random.shuffle(theta)
    return theta


def generate_water_like_dataset(n, mode="ID"):
    """Generate 3D water-like coordinates and metadata for ID or OOD modes."""
    if mode == "ID":
        r1 = np.random.normal(1.0, 0.05, size=n)
        r2 = np.random.normal(1.0, 0.05, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(5.0), size=n)
    elif mode == "OOD_highT":
        r1 = np.random.normal(1.0, 0.18, size=n)
        r2 = np.random.normal(1.0, 0.18, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(20.0), size=n)
    elif mode == "OOD_stretch":
        r1 = np.random.uniform(1.3, 1.8, size=n)
        r2 = np.random.normal(1.0, 0.06, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(6.0), size=n)
    elif mode == "OOD_compress":
        r1 = np.random.uniform(0.65, 0.85, size=n)
        r2 = np.random.normal(1.0, 0.06, size=n)
        theta = np.random.normal(np.deg2rad(104.5), np.deg2rad(6.0), size=n)
    elif mode == "OOD_angle":
        r1 = np.random.normal(1.0, 0.06, size=n)
        r2 = np.random.normal(1.0, 0.06, size=n)
        theta = _sample_angle_ood(n)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    r1 = np.clip(r1, 0.45, 2.2)
    r2 = np.clip(r2, 0.45, 2.2)
    theta = np.clip(theta, np.deg2rad(35), np.deg2rad(175))

    coords = np.zeros((n, 3, 3), dtype=np.float32)
    coords[:, 0, :] = 0.0
    coords[:, 1, 0] = r1 * np.cos(-theta / 2.0)
    coords[:, 1, 1] = r1 * np.sin(-theta / 2.0)
    coords[:, 2, 0] = r2 * np.cos(theta / 2.0)
    coords[:, 2, 1] = r2 * np.sin(theta / 2.0)
    # z coordinates are initially zero: a planar molecule embedded in 3D Cartesian space.
    return coords


def internal_coords_numpy(coords):
    """Compute internal coordinates as numpy arrays for plotting and tables."""
    with torch.no_grad():
        t = torch.tensor(coords, dtype=DTYPE)
        r1, r2, theta = compute_internal_coords_torch(t)
    return r1.numpy(), r2.numpy(), theta.numpy()








这里生成训练集、ID 测试集和四类 OOD 数据。训练集只覆盖接近平衡的水分子构型；四类 OOD 分别测试高温扰动、键拉伸、键压缩和键角偏离时模型是否仍然可靠。




In [ ]:
# Generate train, ID test, and OOD test sets.
train_coords = generate_water_like_dataset(500, "ID")
id_test_coords = generate_water_like_dataset(100, "ID")
ood_modes = ["OOD_highT", "OOD_stretch", "OOD_compress", "OOD_angle"]
ood_coords_by_mode = {m: generate_water_like_dataset(100, m) for m in ood_modes}
ood_coords = np.concatenate([ood_coords_by_mode[m] for m in ood_modes], axis=0)

all_coords = np.concatenate([train_coords, id_test_coords, ood_coords], axis=0)
all_energy, all_forces = reference_energy_and_forces(all_coords)

n_train = len(train_coords)
n_id = len(id_test_coords)
train_energy = all_energy[:n_train]
train_forces = all_forces[:n_train]
id_test_energy = all_energy[n_train:n_train+n_id]
id_test_forces = all_forces[n_train:n_train+n_id]
ood_energy = all_energy[n_train+n_id:]
ood_forces = all_forces[n_train+n_id:]

splits = ["train"] * n_train + ["ID_test"] * n_id + ["OOD_test"] * len(ood_coords)
ood_types = ["ID"] * (n_train + n_id) + sum(([m] * 100 for m in ood_modes), [])
r1_all, r2_all, theta_all = internal_coords_numpy(all_coords)
metadata = pd.DataFrame({
    "structure_id": np.arange(len(all_coords)),
    "split": splits,
    "ood_type": ood_types,
    "r1": r1_all,
    "r2": r2_all,
    "theta_rad": theta_all,
    "theta_deg": np.rad2deg(theta_all),
    "energy": all_energy,
})
metadata.head()








先观察数据覆盖范围。左边是由 r1、r2 和 theta 构成的三维构型空间；右边用于显示选中水分子的球棍结构。3D 图和分子视图加载可能需要一点时间；如果右侧分子没有显示，可以随便点击一个数据点触发刷新。




In [ ]:
plot_meta = metadata.copy()
plot_meta["category"] = np.where(plot_meta["split"].eq("train"), "train", plot_meta["ood_type"])
plot_meta.loc[plot_meta["split"].eq("ID_test"), "category"] = "ID_test"
category_order = ["train", "ID_test", "OOD_highT", "OOD_stretch", "OOD_compress", "OOD_angle"]
color_map = {
    "train": "#1f77b4",
    "ID_test": "#2ca02c",
    "OOD_highT": "#d62728",
    "OOD_stretch": "#9467bd",
    "OOD_compress": "#ff7f0e",
    "OOD_angle": "#e377c2",
}

# py3Dmol normally loads 3Dmol.js from a CDN. VS Code may block that request,
# so we use the vendored local JS bundle as a data URL when it is available.
THREEDMOL_JS_PATH = Path("assets/3Dmol-min.js")
if THREEDMOL_JS_PATH.exists():
    THREEDMOL_JS_URI = "data:text/javascript;base64," + base64.b64encode(THREEDMOL_JS_PATH.read_bytes()).decode("ascii")
else:
    THREEDMOL_JS_URI = "https://cdn.jsdelivr.net/npm/3dmol@2.5.4/build/3Dmol-min.js"


def coords_to_xyz_block(coords, title="water toy"):
    """Convert one 3-atom water-like structure to XYZ text for py3Dmol."""
    lines = ["3", title]
    for element, xyz in zip(["O", "H", "H"], coords):
        lines.append(f"{element} {xyz[0]:.6f} {xyz[1]:.6f} {xyz[2]:.6f}")
    return "\n".join(lines)


def show_py3dmol_structure(structure_id, width=360, height=520):
    """Show one selected structure with py3Dmol ball-and-stick rendering."""
    sid = int(structure_id)
    coords = all_coords[sid]
    row = metadata.loc[metadata["structure_id"].eq(sid)].iloc[0]
    title = f"structure_id={sid}, {row['ood_type']}"
    view = py3Dmol.view(width=width, height=height, js=THREEDMOL_JS_URI)
    view.addModel(coords_to_xyz_block(coords, title=title), "xyz")
    view.setStyle({"stick": {"radius": 0.10}, "sphere": {"radius": 0.40}})
    view.zoomTo()
    view.show()


def write_config_space_html(
    fig,
    metadata_df,
    coords_array,
    filename="config_space_molecule_viewer.html",
    page_title="Configuration-space molecule viewer",
    description="点击左侧 3D 构型空间中的任意点，右侧会显示对应三维水分子结构。",
    initial_structure_id=None,
):
    """Write a standalone browser-viewable HTML for configuration-space exploration."""
    html_path = OUTPUT_DIR / filename
    plot_html = fig.to_html(include_plotlyjs=True, full_html=False, div_id="config-space-plot")
    coords_payload = json.dumps(coords_array.tolist())
    meta_payload = json.dumps(
        metadata_df[["structure_id", "split", "ood_type", "r1", "r2", "theta_deg"]].to_dict(orient="records"),
        ensure_ascii=False,
    )
    if THREEDMOL_JS_PATH.exists():
        threedmol_script = THREEDMOL_JS_PATH.read_text(encoding="utf-8", errors="ignore")
        threedmol_loader = f"<script>{threedmol_script}</script>"
    else:
        threedmol_loader = '<script src="https://cdn.jsdelivr.net/npm/3dmol@2.5.4/build/3Dmol-min.js"></script>'

    html = f"""<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8" />
  <title>MLIP Demo: {page_title}</title>
  <style>
    body {{ margin: 0; font-family: Arial, sans-serif; color: #243b5a; }}
    .page {{ padding: 18px 22px; }}
    h1 {{ margin: 0 0 6px; font-size: 22px; font-weight: 700; }}
    p {{ margin: 0 0 14px; font-size: 14px; }}
    .layout {{ display: grid; grid-template-columns: minmax(620px, 1fr) 390px; gap: 18px; align-items: stretch; }}
    #config-space-plot {{ width: 100%; height: 680px; }}
    .viewer {{ border: 1px solid #d6dbe5; border-radius: 6px; padding: 12px; }}
    #mol-title {{ font-size: 15px; font-weight: 700; margin-bottom: 10px; }}
    #molecule-3dmol {{ width: 100%; height: 610px; position: relative; }}
    @media (max-width: 980px) {{ .layout {{ grid-template-columns: 1fr; }} #config-space-plot {{ height: 560px; }} }}
  </style>
</head>
<body>
  <div class="page">
    <h1>{page_title}</h1>
    <p>{description}</p>
    <div class="layout">
      <div>{plot_html}</div>
      <div class="viewer">
        <div id="mol-title">Selected molecule</div>
        <div id="molecule-3dmol"></div>
      </div>
    </div>
  </div>
  {threedmol_loader}
  <script>
    const coordsData = {coords_payload};
    const metaData = {meta_payload};
    const metaById = new Map(metaData.map(row => [Number(row.structure_id), row]));

    function xyzBlock(structureId) {{
      const coords = coordsData[structureId];
      const row = metaById.get(Number(structureId));
      const title = `structure_id=${{structureId}}, ${{row.ood_type}}`;
      const lines = ['3', title];
      const elements = ['O', 'H', 'H'];
      for (let i = 0; i < 3; i++) {{
        const xyz = coords[i];
        lines.push(`${{elements[i]}} ${{xyz[0].toFixed(6)}} ${{xyz[1].toFixed(6)}} ${{xyz[2].toFixed(6)}}`);
      }}
      return lines.join(String.fromCharCode(10)) + String.fromCharCode(10);
    }}

    function get3DmolLib() {{
      const molLib = window.$3Dmol || window["3Dmol"];
      if (molLib && !window.$3Dmol) {{
        window.$3Dmol = molLib;
      }}
      return molLib;
    }}

    function showMolecule(structureId) {{
      const sid = Number(structureId);
      const row = metaById.get(sid);
      const div = document.getElementById('molecule-3dmol');
      document.getElementById('mol-title').textContent =
        `structure_id=${{sid}} | ${{row.split}} | ${{row.ood_type}} | r1=${{row.r1.toFixed(3)}} | r2=${{row.r2.toFixed(3)}} | theta=${{row.theta_deg.toFixed(1)}}°`;

      window.requestAnimationFrame(() => {{
        try {{
          const molLib = get3DmolLib();
          if (!molLib || !molLib.createViewer) {{
            div.innerHTML = '<div style="padding:12px;color:#b00020;font-weight:700;">3Dmol.js 未成功加载，请检查浏览器控制台。</div>';
            return;
          }}
          div.innerHTML = '';
          div.style.width = '100%';
          div.style.height = '610px';
          const viewer = molLib.createViewer(div, {{ backgroundColor: 'white' }});
          const model = viewer.addModel(xyzBlock(sid), 'xyz', {{ assignBonds: true }});
          viewer.setStyle({{ stick: {{ radius: 0.10 }}, sphere: {{ radius: 0.40 }} }});
          viewer.zoomTo();
          viewer.render();
          window.setTimeout(() => {{
            if (viewer.resize) viewer.resize();
            viewer.render();
          }}, 100);
        }} catch (err) {{
          console.error(err);
          div.innerHTML = `<div style="padding:12px;color:#b00020;font-weight:700;">分子渲染失败：${{err.message}}</div>`;
        }}
      }});
    }}

    function initInteractions() {{
      const plotDiv = document.getElementById('config-space-plot');
      if (plotDiv && plotDiv.on) {{
        plotDiv.on('plotly_click', event => {{
          const point = event.points && event.points[0];
          if (!point || !point.customdata) return;
          showMolecule(Number(point.customdata[0]));
        }});
      }}
      const initialStructureId = {json.dumps(int(initial_structure_id) if initial_structure_id is not None else None)};
      showMolecule(initialStructureId ?? metaData[0].structure_id);
    }}

    if (document.readyState === 'loading') {{
      document.addEventListener('DOMContentLoaded', () => window.setTimeout(initInteractions, 0));
    }} else {{
      window.setTimeout(initInteractions, 0);
    }}
  </script>
</body>
</html>
"""
    html_path.write_text(html, encoding="utf-8")
    print(f"Saved interactive HTML: {html_path}")
    return html_path


def axis_range_with_padding(values, pad_fraction=0.05):
    """Return a fixed Plotly axis range with a small visual padding."""
    lo = float(np.nanmin(values))
    hi = float(np.nanmax(values))
    pad = max((hi - lo) * pad_fraction, 1e-6)
    return [lo - pad, hi + pad]

config_axis_ranges = {
    "x": axis_range_with_padding(plot_meta["r1"]),
    "y": axis_range_with_padding(plot_meta["r2"]),
    "z": axis_range_with_padding(plot_meta["theta_deg"]),
}

fig_widget = go.FigureWidget()
for category in category_order:
    group = plot_meta[plot_meta["category"] == category]
    fig_widget.add_trace(go.Scatter3d(
        x=group["r1"],
        y=group["r2"],
        z=group["theta_deg"],
        mode="markers",
        name=category,
        customdata=group[["structure_id", "split", "ood_type"]].values,
        marker={
            "size": 4 if category != "train" else 3,
            "color": color_map[category],
            "opacity": 0.80,
        },
        hovertemplate=(
            "id=%{customdata[0]}<br>"
            "split=%{customdata[1]}<br>"
            "type=%{customdata[2]}<br>"
            "r1=%{x:.3f}<br>r2=%{y:.3f}<br>theta=%{z:.1f} deg"
            "<extra></extra>"
        ),
    ))

fig_widget.update_layout(
    title={"text": "<b>Configuration space coverage</b>", "x": 0.5, "xanchor": "center", "y": 0.96},
    font={"family": "Arial", "size": 13, "color": "#243b5a"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "center", "x": 0.5},
    scene={
        "xaxis": {"title": "r1 = |H1 - O|", "range": config_axis_ranges["x"], "autorange": False},
        "yaxis": {"title": "r2 = |H2 - O|", "range": config_axis_ranges["y"], "autorange": False},
        "zaxis": {"title": "theta (degree)", "range": config_axis_ranges["z"], "autorange": False},
    },
    margin={"l": 0, "r": 0, "b": 0, "t": 115},
    width=850,
    height=650,
)

write_config_space_html(fig_widget, metadata, all_coords)

mol_output = widgets.Output(layout={"width": "390px", "height": "620px", "border": "1px solid #ddd"})
status = widgets.HTML(value="<b>Selected molecule</b>: click a point in the Plotly panel.")
right_panel = widgets.VBox([status, mol_output], layout=widgets.Layout(width="410px"))


def update_selected_molecule(structure_id):
    """Refresh the py3Dmol output panel for a clicked structure."""
    sid = int(structure_id)
    row = metadata.loc[metadata["structure_id"].eq(sid)].iloc[0]
    status.value = f"<b>Selected molecule</b>: structure_id={sid}, type={row['ood_type']}"
    with mol_output:
        clear_output(wait=True)
        show_py3dmol_structure(sid, width=360, height=520)


def handle_click(trace, points, selector):
    """Plotly FigureWidget click callback: update the py3Dmol molecule panel."""
    if not points.point_inds:
        return
    sid = trace.customdata[points.point_inds[0]][0]
    update_selected_molecule(sid)


for trace in fig_widget.data:
    trace.on_click(handle_click)

first_id = int(plot_meta.loc[plot_meta["category"].eq("OOD_angle"), "structure_id"].iloc[0])
# Fill the molecule panel before displaying the HBox, so the first render is not blank in VS Code.
update_selected_molecule(first_id)
display(widgets.HTML("<b>Interactive 3D view.</b> Loading may take a few seconds. If the molecule panel is blank, click any data point to refresh it."))
display(widgets.HBox([fig_widget, right_panel], layout=widgets.Layout(align_items="stretch")))












## Section 2. 模型模块：对比原始坐标与物理不变特征

这一节训练两个 toy MLIP，用来展示表示方式对可靠性的影响。

- **RawCoordNet** 直接使用 9 个笛卡尔坐标作为输入，模型可以拟合训练集，但容易依赖训练时的坐标系。
- **InvariantFeatureNet** 使用 `[sort(r1, r2), theta]` 作为输入，显式加入平移、旋转和同种 H 置换不变性。

训练目标同时包含能量 MSE 和力 MSE，所以模型要同时拟合势能面的值和梯度。这里的重点不是比较网络大小，而是比较“是否把物理对称性放进表示里”。



这里定义两个模型进行对比：RawCoordNet 直接使用笛卡尔坐标，InvariantFeatureNet 使用键长和角度等不变特征。对比重点是物理对称性对模型行为的影响。




In [ ]:
class RawCoordNet(nn.Module):
    """MLP that predicts energy directly from flattened 3D Cartesian coordinates."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(9, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 1),
        )

    def forward(self, coords):
        x = coords.reshape(coords.shape[0], -1)
        return self.net(x).squeeze(-1)


class InvariantFeatureNet(nn.Module):
    """MLP that predicts energy from rotation/translation invariant features."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 1),
        )

    def forward(self, coords):
        r1, r2, theta = compute_internal_coords_torch(coords, sort_h=True)
        x = torch.stack([r1, r2, theta], dim=-1)
        return self.net(x).squeeze(-1)


def set_seed(seed):
    """Set random seeds for reproducible training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_energy_model(model, coords_np, energy_np, forces_np, epochs=300, batch_size=64, lr=5e-4, seed=0, force_weight=1.0, sample_weights=None):
    """Train a neural potential with energy and force losses."""
    set_seed(seed)
    model.to(DEVICE)
    coords = torch.tensor(coords_np, dtype=DTYPE)
    energy = torch.tensor(energy_np, dtype=DTYPE)
    forces = torch.tensor(forces_np, dtype=DTYPE)
    if sample_weights is None:
        weights = torch.ones(len(coords), dtype=DTYPE)
    else:
        weights = torch.tensor(sample_weights, dtype=DTYPE)
        weights = weights / weights.mean()
    loader = DataLoader(TensorDataset(coords, energy, forces, weights), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        total = 0.0
        for xb, yb, fb, wb in loader:
            xb = xb.detach().clone().requires_grad_(True)
            optimizer.zero_grad()
            pred = model(xb)
            grad = torch.autograd.grad(pred.sum(), xb, create_graph=True)[0]
            pred_forces = -grad
            energy_loss = torch.mean(wb * (pred - yb)**2)
            force_loss = torch.mean(wb[:, None, None] * (pred_forces - fb)**2)
            loss = energy_loss + force_weight * force_loss
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
        history.append(total / len(coords))
    return history


def energy_mae(model, coords_np, energy_np):
    """Compute energy MAE."""
    model.eval()
    with torch.no_grad():
        coords = torch.tensor(coords_np, dtype=DTYPE)
        pred = model(coords).cpu().numpy()
    return np.mean(np.abs(pred - energy_np))








这里开始训练两个势函数。loss 里同时包含能量和力，所以模型要同时拟合势能面的值和梯度。




In [ ]:
raw_model = RawCoordNet()
inv_model = InvariantFeatureNet()

raw_history = train_energy_model(raw_model, train_coords, train_energy, train_forces, epochs=500, seed=1)
inv_history = train_energy_model(inv_model, train_coords, train_energy, train_forces, epochs=500, seed=2)

raw_id_mae = energy_mae(raw_model, id_test_coords, id_test_energy)
inv_id_mae = energy_mae(inv_model, id_test_coords, id_test_energy)

print(f"RawCoordNet ID test energy MAE:       {raw_id_mae:.5f}")
print(f"InvariantFeatureNet ID energy MAE:    {inv_id_mae:.5f}")








这张图用于检查训练是否正常收敛。两个模型的 loss 不需要完全一致，后续会进一步比较它们在对称性和 OOD 测试中的表现。




In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.plot(raw_history, label="RawCoordNet", color=MODEL_COLORS["RawCoordNet"])
ax.plot(inv_history, label="InvariantFeatureNet", color=MODEL_COLORS["InvariantFeatureNet"])
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("Energy + force loss")
ax.set_title("Training loss", pad=10)
ax.grid(axis="y", which="major", alpha=0.25)
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
save_figure(fig, "01_training_loss.png")
plt.show()










## Section 3. 预测模块：从能量自动微分得到力

MLIP 通常先预测势能面，再通过能量梯度得到力：

$$
F_i = -\frac{\partial E}{\partial R_i}
$$

这一节把模型预测的能量转化为力，并在 ID 与各类 OOD 构型上统计 energy RMSE/MAE 和 force RMSE/MAE。核心观察是：ID 上的误差低，并不自动意味着 OOD 区域可靠。




这里先用模型预测能量，再通过 autograd 对坐标求梯度得到预测力。表格和柱状图用于比较 ID 与 OOD 条件下的能量和力误差。




In [ ]:
def predict_energy_and_forces(model, coords_np):
    """Predict energies and forces from a differentiable energy model."""
    model.eval()
    coords = torch.tensor(coords_np, dtype=DTYPE, requires_grad=True)
    energy = model(coords)
    grad = torch.autograd.grad(energy.sum(), coords, create_graph=False)[0]
    forces = -grad
    return energy.detach().numpy(), forces.detach().numpy()


def force_mae(pred_forces, ref_forces):
    """Mean absolute error over all force components."""
    return np.mean(np.abs(pred_forces - ref_forces))


def rmse(pred, ref):
    """Root mean squared error over all entries."""
    return float(np.sqrt(np.mean((pred - ref)**2)))


def evaluate_model_by_group(model, name):
    """Evaluate energy and force MAE for ID test and each OOD group."""
    rows = []
    groups = [("ID_test", id_test_coords, id_test_energy, id_test_forces)]
    start = 0
    for mode in ood_modes:
        end = start + 100
        groups.append((mode, ood_coords[start:end], ood_energy[start:end], ood_forces[start:end]))
        start = end
    for group, coords, e_ref, f_ref in groups:
        e_pred, f_pred = predict_energy_and_forces(model, coords)
        rows.append({
            "model": name,
            "group": group,
            "energy_MAE": np.mean(np.abs(e_pred - e_ref)),
            "energy_RMSE": rmse(e_pred, e_ref),
            "force_MAE": force_mae(f_pred, f_ref),
            "force_RMSE": rmse(f_pred, f_ref),
        })
    return pd.DataFrame(rows)

metrics = pd.concat([
    evaluate_model_by_group(raw_model, "RawCoordNet"),
    evaluate_model_by_group(inv_model, "InvariantFeatureNet"),
], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
groups = ["ID_test"] + ood_modes
x = np.arange(len(groups))
width = 0.34
for offset, model_name in [(-width/2, "RawCoordNet"), (width/2, "InvariantFeatureNet")]:
    sub = metrics[metrics["model"] == model_name].set_index("group").loc[groups]
    axes[0].bar(x + offset, sub["energy_RMSE"], width, label=model_name, color=MODEL_COLORS[model_name])
    axes[1].bar(x + offset, sub["force_RMSE"], width, label=model_name, color=MODEL_COLORS[model_name])
for ax, ylabel, title in [
    (axes[0], "RMSE", "RMSE of Energy"),
    (axes[1], "RMSE", "RMSE of Force"),
]:
    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=25, ha="right", fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    ax.set_yscale("log")
    ax.grid(axis="y", which="major", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle("ID/OOD error by model", y=0.99)
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.93), ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.82], w_pad=2.4)
save_figure(fig, "02_id_ood_rmse_by_model.png")
plt.show()

metrics.round(4)









### 坐标系变换 stress test

这里对同一批 ID test 构型做整体三维旋转和平移。物理构型没有改变，因此参考能量应保持不变，参考力应随同一个旋转矩阵旋转。

这个测试专门用来暴露 RawCoordNet 对坐标系的依赖，也帮助学生区分“拟合训练集”和“满足物理对称性”是两件事。



这是一个坐标变换压力测试：同一批 ID 水分子只经过整体旋转和平移，真实能量不应改变，真实力应随坐标系一起旋转。




In [ ]:
def random_rotation_matrix(seed=0):
    """Create a deterministic random 3D rotation matrix."""
    rng = np.random.default_rng(seed)
    q = rng.normal(size=4)
    q = q / np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*z*w,     2*x*z + 2*y*w],
        [2*x*y + 2*z*w,     1 - 2*x*x - 2*z*z, 2*y*z - 2*x*w],
        [2*x*z - 2*y*w,     2*y*z + 2*x*w,     1 - 2*x*x - 2*y*y],
    ], dtype=np.float32)

R_stress = random_rotation_matrix(seed=2026)
t_stress = np.array([1.3, -0.8, 0.6], dtype=np.float32)
id_test_coords_rt = id_test_coords @ R_stress.T + t_stress
id_test_forces_rt = id_test_forces @ R_stress.T

stress_rows = []
for model, name in [(raw_model, "RawCoordNet"), (inv_model, "InvariantFeatureNet")]:
    e_pred, f_pred = predict_energy_and_forces(model, id_test_coords_rt)
    stress_rows.append({
        "model": name,
        "test_set": "same ID structures, rotated + translated",
        "energy_MAE": np.mean(np.abs(e_pred - id_test_energy)),
        "energy_RMSE": rmse(e_pred, id_test_energy),
        "force_MAE": force_mae(f_pred, id_test_forces_rt),
        "force_RMSE": rmse(f_pred, id_test_forces_rt),
    })
stress_metrics = pd.DataFrame(stress_rows)

fig, ax = plt.subplots(figsize=(7.4, 4.1))
metric_labels = ["RMSE of Energy", "RMSE of Force"]
x = np.arange(len(metric_labels))
width = 0.34
for offset, model_name in [(-width/2, "RawCoordNet"), (width/2, "InvariantFeatureNet")]:
    row = stress_metrics[stress_metrics["model"] == model_name].iloc[0]
    ax.bar(x + offset, [row["energy_RMSE"], row["force_RMSE"]], width, label=model_name, color=MODEL_COLORS[model_name])
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontweight="bold")
ax.set_ylabel("RMSE")
ax.grid(axis="y", which="major", alpha=0.25)
fig.suptitle("Coordinate transform error", y=0.955)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.885), ncol=2, frameon=False)
if ax.legend_ is not None:
    ax.legend_.remove()
plt.tight_layout(rect=[0, 0, 1, 0.84])
save_figure(fig, "03_coordinate_transform_error.png")
plt.show()

stress_metrics.round(4)









## Section 4. 对称性模块：检查能量不变性与力等变性

物理上，总能量应当对整体平移和旋转不变。力不是旋转不变的向量，而是应当随构型一起旋转，也就是旋转等变：

$$
E(Rx) = E(x), \quad F(Rx) = R F(x)
$$

这一节对 RawCoordNet 和 InvariantFeatureNet 分别做平移、旋转和力等变性检查。这个 sanity check 可以在真正跑大规模模拟前，提前发现表示方式中的明显物理问题。



这里检查物理对称性。能量应满足平移和旋转不变性；力不是旋转不变的，而应随分子构型旋转，即满足旋转等变性。




In [ ]:
def rotation_matrix_3d(axis, angle_rad):
    """Build a 3D rotation matrix from an axis and angle."""
    axis = np.asarray(axis, dtype=np.float32)
    axis = axis / np.linalg.norm(axis)
    x, y, z = axis
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    C = 1.0 - c
    return np.array([
        [c + x*x*C,     x*y*C - z*s, x*z*C + y*s],
        [y*x*C + z*s,   c + y*y*C,   y*z*C - x*s],
        [z*x*C - y*s,   z*y*C + x*s, c + z*z*C],
    ], dtype=np.float32)


def rotate_coords(coords_np, angle_rad):
    """Rotate 3D coordinates around an arbitrary 3D axis by angle_rad."""
    r = rotation_matrix_3d(axis=[1.0, 1.0, 0.5], angle_rad=angle_rad)
    return coords_np @ r.T


def symmetry_checks(model, coords_single):
    """Compute translation invariance, rotation invariance, and force equivariance errors."""
    x = coords_single[None, :, :]
    translation = np.array([0.7, -1.2, 0.4], dtype=np.float32)
    xt = x + translation
    angle = np.deg2rad(60.0)
    xr = rotate_coords(x, angle)

    e, f = predict_energy_and_forces(model, x)
    et, _ = predict_energy_and_forces(model, xt)
    er, fr = predict_energy_and_forces(model, xr)
    rf = rotate_coords(f, angle)

    return {
        "translation_energy_error": float(np.abs(et[0] - e[0])),
        "rotation_energy_error": float(np.abs(er[0] - e[0])),
        "force_equivariance_error": float(np.linalg.norm(fr[0] - rf[0], axis=1).mean()),
    }

x0 = id_test_coords[0]
symmetry_table = pd.DataFrame([
    {"model": "RawCoordNet", **symmetry_checks(raw_model, x0)},
    {"model": "InvariantFeatureNet", **symmetry_checks(inv_model, x0)},
])

fig, ax = plt.subplots(figsize=(8.4, 4.1))
symmetry_metrics = ["translation_energy_error", "rotation_energy_error", "force_equivariance_error"]
metric_labels = ["translation E", "rotation E", "force equiv."]
x = np.arange(len(symmetry_metrics))
width = 0.34
for offset, model_name in [(-width/2, "RawCoordNet"), (width/2, "InvariantFeatureNet")]:
    row = symmetry_table[symmetry_table["model"] == model_name].iloc[0]
    values = np.maximum([row[m] for m in symmetry_metrics], 1e-12)
    ax.bar(x + offset, values, width, label=model_name, color=MODEL_COLORS[model_name])
ax.set_yscale("log")
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontweight="bold")
ax.set_ylabel("Error")
ax.grid(axis="y", which="major", alpha=0.25)
fig.suptitle("Symmetry error check", y=0.955)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.885), ncol=2, frameon=False)
if ax.legend_ is not None:
    ax.legend_.remove()
plt.tight_layout(rect=[0, 0, 1, 0.84])
save_figure(fig, "04_symmetry_error_check.png")
plt.show()

symmetry_table.round(6)









## Section 5. 诊断模块：识别 OOD 失效区域

这一节聚焦对称性更合理的 InvariantFeatureNet，比较它在 ID 和四类 OOD 构型上的误差，并找出 force RMSE 最大的结构。

我们把高误差构型重新放回构型空间中观察：如果它们集中在训练数据覆盖不足的区域，就说明问题主要来自数据盲区，而不只是模型没有训练好。



这里聚焦 InvariantFeatureNet，将它在 ID 和各类 OOD 构型上的误差分组统计。高误差结构有助于定位模型在哪些构型区域失效。




In [ ]:
def per_structure_force_mae(pred_forces, ref_forces):
    """Compute per-structure force MAE over atoms and coordinates."""
    return np.mean(np.abs(pred_forces - ref_forces), axis=(1, 2))


def per_structure_force_rmse(pred_forces, ref_forces):
    """Compute per-structure force RMSE over atoms and coordinates."""
    return np.sqrt(np.mean((pred_forces - ref_forces)**2, axis=(1, 2)))

inv_eval_metrics = metrics[metrics["model"] == "InvariantFeatureNet"].copy()
inv_eval_metrics








这里先计算每个 OOD 构型的 force MAE 和 force RMSE，并找出误差最大的结构。下一张图会把这些高误差点放回构型空间中观察。




In [ ]:
e_ood_pred, f_ood_pred = predict_energy_and_forces(inv_model, ood_coords)
per_ood_force_mae = per_structure_force_mae(f_ood_pred, ood_forces)
per_ood_force_rmse = per_structure_force_rmse(f_ood_pred, ood_forces)
ood_meta = metadata[metadata["split"] == "OOD_test"].copy().reset_index(drop=True)
ood_meta["force_mae"] = per_ood_force_mae
ood_meta["force_rmse"] = per_ood_force_rmse

top10 = (ood_meta.sort_values("force_rmse", ascending=False)
         [["structure_id", "ood_type", "r1", "r2", "theta_deg", "force_mae", "force_rmse"]]
         .head(10))

print("Computed per-structure OOD force errors for diagnostics.")








这里将高误差 OOD 构型映射回构型空间。下方的 3D 图可以点击数据点并在右侧显示对应分子结构；3D 图和分子视图加载可能需要一点时间，如果右侧没有显示，可以点击任意数据点触发刷新。




In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 5.4))
train_meta = metadata[metadata["split"] == "train"]
id_meta = metadata[metadata["split"] == "ID_test"]
high_error_ids = set(top10["structure_id"])
high_error = ood_meta[ood_meta["structure_id"].isin(high_error_ids)].copy()
other_ood = ood_meta[~ood_meta["structure_id"].isin(high_error_ids)].copy()

ax.scatter(train_meta["r1"], train_meta["theta_deg"], s=14, alpha=0.35, c="#1f77b4", label="train ID")
ax.scatter(id_meta["r1"], id_meta["theta_deg"], s=14, alpha=0.35, c="#2ca02c", label="ID test")
ax.scatter(ood_meta["r1"], ood_meta["theta_deg"], s=12, alpha=0.12, c="#7f7f7f", label="OOD pool")
ax.scatter(high_error["r1"], high_error["theta_deg"], s=72, c="#d62728", edgecolors="white", linewidths=0.8, label="top error OOD")
ax.set_xlabel("r1")
ax.set_ylabel("theta (degree)")
fig.suptitle("High-error OOD locations", y=0.955)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.885), ncol=4, frameon=False, fontsize=9)
if ax.legend_ is not None:
    ax.legend_.remove()
plt.tight_layout(rect=[0, 0, 1, 0.84])
save_figure(fig, "05_high_error_ood_locations.png")
plt.show()

ood_axis_df = pd.concat([train_meta, id_meta, ood_meta], ignore_index=True)
ood_axis_ranges = {
    "x": axis_range_with_padding(ood_axis_df["r1"]),
    "y": axis_range_with_padding(ood_axis_df["r2"]),
    "z": axis_range_with_padding(ood_axis_df["theta_deg"]),
}

fig3d = go.FigureWidget()

def add_config_trace(fig, df, name, color, opacity, size, include_force=False):
    """Add one clickable configuration-space trace to the Plotly widget."""
    custom_cols = ["structure_id", "ood_type"] + (["force_rmse"] if include_force else [])
    hover = (
        "id=%{customdata[0]}<br>type=%{customdata[1]}<br>"
        "r1=%{x:.3f}<br>r2=%{y:.3f}<br>theta=%{z:.1f} deg"
    )
    if include_force:
        hover += "<br>force RMSE=%{customdata[2]:.3f}"
    hover += "<extra></extra>"
    fig.add_trace(go.Scatter3d(
        x=df["r1"],
        y=df["r2"],
        z=df["theta_deg"],
        mode="markers",
        name=name,
        customdata=df[custom_cols].values,
        marker={"size": size, "color": color, "opacity": opacity},
        hovertemplate=hover,
    ))

# Context points first: same marker size for train / ID test / other OOD.
add_config_trace(fig3d, train_meta.assign(ood_type="train ID"), "train ID", "#1f77b4", 0.45, 3, include_force=False)
add_config_trace(fig3d, id_meta.assign(ood_type="ID test"), "ID test", "#2ca02c", 0.45, 3, include_force=False)
add_config_trace(fig3d, other_ood, "other OOD", "#9e9e9e", 0.35, 3, include_force=True)

# Selected points last, with larger opaque red markers, so they are easier to hover/click.
fig3d.add_trace(go.Scatter3d(
    x=high_error["r1"],
    y=high_error["r2"],
    z=high_error["theta_deg"],
    mode="markers",
    name="top error OOD",
    customdata=high_error[["structure_id", "ood_type", "force_rmse"]].values,
    marker={"size": 7, "color": "#d62728", "opacity": 1.0, "line": {"color": "white", "width": 1}},
    hovertemplate=(
        "id=%{customdata[0]}<br>type=%{customdata[1]}<br>"
        "r1=%{x:.3f}<br>r2=%{y:.3f}<br>theta=%{z:.1f} deg<br>"
        "force RMSE=%{customdata[2]:.3f}<extra></extra>"
    ),
))
fig3d.update_layout(
    title={"text": "<b>Selected high-error OOD points</b>", "x": 0.5, "xanchor": "center", "y": 0.96},
    font={"family": "Arial", "size": 13, "color": "#243b5a"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "xanchor": "center", "x": 0.5},
    scene={
        "xaxis": {"title": "r1", "range": ood_axis_ranges["x"], "autorange": False},
        "yaxis": {"title": "r2", "range": ood_axis_ranges["y"], "autorange": False},
        "zaxis": {"title": "theta (degree)", "range": ood_axis_ranges["z"], "autorange": False},
    },
    margin={"l": 0, "r": 0, "b": 0, "t": 115},
    width=850,
    height=650,
)

first_high_error_id = int(high_error.sort_values("force_rmse", ascending=False).iloc[0]["structure_id"])
write_config_space_html(
    fig3d,
    metadata,
    all_coords,
    filename="high_error_ood_molecule_viewer.html",
    page_title="High-error OOD molecule viewer",
    description="点击左侧 3D 构型空间中的任意点，右侧会显示对应三维水分子结构。红色点是 force RMSE 最高的 OOD 构型。",
    initial_structure_id=first_high_error_id,
)

ood_mol_output = widgets.Output(layout={"width": "390px", "height": "620px", "border": "1px solid #ddd"})
ood_status = widgets.HTML(value="<b>Selected molecule</b>: click a point in the 3D plot.")
ood_right_panel = widgets.VBox([ood_status, ood_mol_output], layout=widgets.Layout(width="410px"))


def update_ood_molecule(structure_id):
    """Refresh the molecule panel for the clicked structure in the OOD diagnostic plot."""
    sid = int(structure_id)
    row = metadata.loc[metadata["structure_id"].eq(sid)].iloc[0]
    ood_status.value = f"<b>Selected molecule</b>: structure_id={sid}, type={row['ood_type']}"
    with ood_mol_output:
        clear_output(wait=True)
        show_py3dmol_structure(sid, width=360, height=520)


def handle_ood_click(trace, points, selector):
    """Update molecule rendering when a point is clicked in the OOD 3D plot."""
    if not points.point_inds:
        return
    sid = trace.customdata[points.point_inds[0]][0]
    update_ood_molecule(sid)


for trace in fig3d.data:
    trace.on_click(handle_ood_click)

update_ood_molecule(first_high_error_id)
display(widgets.HTML("<b>Interactive 3D view.</b> Loading may take a few seconds. If the molecule panel is blank, click any data point to refresh it."))
display(widgets.HBox([fig3d, ood_right_panel], layout=widgets.Layout(align_items="stretch")))











## Section 6. 主动学习模块：用 committee disagreement 选点

主动学习的思想是：如果 DFT 标注很贵，不应随机补数据，而应优先选择模型最不确定、最有信息量的构型。

这里训练 4 个不同随机种子的 InvariantFeatureNet，假装 OOD pool 没有标签。对每个 pool 构型，我们计算 4 个模型预测力的分歧：

$$
uncertainty = mean(std(F_{committee}))
$$

为了做课堂对照，本节也使用 toy 数据里可见的伪 DFT 标签，按真实 force RMSE 做 oracle selection：每个 OOD 区域各选误差最大的 5 个构型，总共 20 个点。真实主动学习不能提前知道 force RMSE，通常用 committee disagreement 近似这个目标。



这里模拟主动学习中的 committee disagreement。多个模型对同一构型的力预测分歧越大，说明该构型的不确定性越高，越值得优先补充标注。




In [ ]:
committee = [inv_model]
committee_histories = [inv_history]
for seed in [10, 11, 12]:
    m = InvariantFeatureNet()
    hist = train_energy_model(m, train_coords, train_energy, train_forces, epochs=250, seed=seed)
    committee.append(m)
    committee_histories.append(hist)

force_predictions = []
energy_predictions = []
for m in committee:
    ep, fp = predict_energy_and_forces(m, ood_coords)
    energy_predictions.append(ep)
    force_predictions.append(fp)
force_predictions = np.stack(force_predictions, axis=0)  # (n_models, n_structures, 3, 3)
uncertainty = force_predictions.std(axis=0).mean(axis=(1, 2))

pool_table = ood_meta.copy()
pool_table["uncertainty"] = uncertainty

N_ORACLE_PER_OOD_TYPE = 5
selected = (pool_table.sort_values("force_rmse", ascending=False)
            .groupby("ood_type", group_keys=False)
            .head(N_ORACLE_PER_OOD_TYPE)
            .sort_values(["ood_type", "force_rmse"], ascending=[True, False]))
print(f"Selected {len(selected)} oracle-label structures for the active-learning round.")








## Section 7. 闭环模块：进行一次主动学习数据回流

这一节把上一步选出的 OOD 构型加入训练集，并从头训练一个新的 InvariantFeatureNet。

主动学习的价值不是简单增加数据量，而是把标注预算优先用在模型最不可靠、最有信息量的区域。最后的对比图展示数据回流前后 OOD force RMSE 的变化。



最后进行一次数据回流：从每类 OOD 中选择需要补充标注的构型，加入训练集后从头训练，并比较 OOD force RMSE 的变化。




In [ ]:
selected_idx = selected.index.to_numpy()
augmented_coords = np.concatenate([train_coords, ood_coords[selected_idx]], axis=0)
augmented_energy = np.concatenate([train_energy, ood_energy[selected_idx]], axis=0)
augmented_forces = np.concatenate([train_forces, ood_forces[selected_idx]], axis=0)
sample_weights = np.ones(len(augmented_coords), dtype=np.float32)
sample_weights[-len(selected_idx):] = 1.0  # conservative: avoid overfitting to only 10 new labels

al_model = InvariantFeatureNet()
al_history = train_energy_model(al_model, augmented_coords, augmented_energy, augmented_forces, epochs=500, seed=123, sample_weights=sample_weights)
al_metrics = evaluate_model_by_group(al_model, "InvariantFeatureNet + 20 oracle AL points")

before_after = pd.concat([
    inv_eval_metrics.assign(model="before"),
    al_metrics.assign(model="after"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(8.8, 4.2))
x = np.arange(len(inv_eval_metrics["group"]))
width = 0.34
before = before_after[before_after["model"] == "before"]
after = before_after[before_after["model"] == "after"]
ax.bar(x - width/2, before["force_RMSE"], width, label="before", color="#7f7f7f")
ax.bar(x + width/2, after["force_RMSE"], width, label="after + 20 labels", color="#2ca02c")
ax.set_xticks(x)
ax.set_xticklabels(before["group"], rotation=25, ha="right", fontweight="bold")
ax.set_ylabel("RMSE of Force")
ax.set_yscale("log")
ax.grid(axis="y", which="major", alpha=0.25)
fig.suptitle("Active-learning update", y=0.955)
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.885), ncol=2, frameon=False)
if ax.legend_ is not None:
    ax.legend_.remove()
plt.tight_layout(rect=[0, 0, 1, 0.84])
save_figure(fig, "06_active_learning_update.png")
plt.show()

display(al_metrics.round(4))
before_mean = inv_eval_metrics[inv_eval_metrics["group"] != "ID_test"]["force_RMSE"].mean()
after_mean = al_metrics[al_metrics["group"] != "ID_test"]["force_RMSE"].mean()
print(f"Mean OOD force RMSE: before = {before_mean:.4f}, after = {after_mean:.4f}")
print("主动学习不是盲目加数据，而是优先补充模型最不确定、最有信息量的构型。")







## 总结：我们学到了什么

1. **MLIP 学习的是构型到能量的函数。** 模型先预测能量，再通过能量梯度得到力，因此力误差反映了势能面局部斜率是否可靠。
2. **物理对称性是必要的 sanity check。** 能量应满足平移、旋转和同种原子置换不变性；力应满足旋转等变性。
3. **ID 测试误差低不代表 OOD 可靠。** 当构型进入训练数据覆盖不足的区域，模型可能给出低可信度的力和能量。
4. **主动学习关注数据盲区。** Committee disagreement 可以帮助优先选择值得补充 DFT 标注的构型，形成持续验证与数据回流闭环。

可靠的机器学习势函数，不只依赖模型架构，还依赖训练数据对目标构型空间的覆盖，以及持续的验证与数据回流。

